# Lesson 4: SVM 元音分类器教程

本教程演示如何使用 SVM (Support Vector Machine) 进行元音分类。

## 学习目标
- 理解 SVM 分类原理
- 学习语音特征提取（共振峰、基频、时长）
- 掌握模型训练和评估流程
- 可视化分类结果

## 1. 环境准备

In [ ]:
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_pipeline.feature_extractor import PraatFeatureExtractor
from src.data_pipeline.dataset_builder import DatasetBuilder
from src.models.svm_classifier import SVMClassifier
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.visualizer import Visualizer

print("环境准备完成！")

## 2. 数据准备

假设我们有以下数据结构：
```
material/lesson_4/
    cantonese_v2/          # 音频文件
    cantonese_v2_out_TG/   # TextGrid 文件
```

In [ ]:
# 配置路径
audio_dir = project_root / 'material' / 'lesson_4' / 'cantonese_v2'
textgrid_dir = project_root / 'material' / 'lesson_4' / 'cantonese_v2_out_TG'
output_dir = project_root / 'results' / 'lesson_04_notebook'
output_dir.mkdir(exist_ok=True, parents=True)

# 目标元音
target_vowels = ['a', 'e', 'i', 'o', 'u']

print(f"音频目录: {audio_dir}")
print(f"TextGrid 目录: {textgrid_dir}")
print(f"目标元音: {target_vowels}")

## 3. 特征提取

使用 Praat 提取语音特征：
- **共振峰 (F1, F2, F3)**: 反映声道共振特性
- **基频 (F0)**: 音高信息
- **时长**: 元音持续时间

In [ ]:
# 初始化特征提取器
config = {
    'praat': {
        'formants': {
            'max_formant': 5500
        },
        'pitch': {
            'time_step': 0.01,
            'pitch_floor': 75,
            'pitch_ceiling': 500
        }
    }
}

extractor = PraatFeatureExtractor(config)

# 示例：提取单个文件的特征
# audio_file = audio_dir / 'example.wav'
# features = extractor.extract_formants(str(audio_file))
# print(f"共振峰: F1={features['F1']:.1f} Hz, F2={features['F2']:.1f} Hz, F3={features['F3']:.1f} Hz")

## 4. 构建数据集

将提取的特征组织成训练集和测试集。

In [ ]:
# 假设已经提取好特征并保存为 CSV
features_file = output_dir / 'features.csv'

# 如果特征文件存在，加载数据
if features_file.exists():
    df = pd.read_csv(features_file)
    print(f"加载特征数据: {len(df)} 个样本")
    print(f"\n特征列: {df.columns.tolist()}")
    print(f"\n前 5 行数据:")
    display(df.head())
    
    # 统计各元音数量
    print(f"\n元音分布:")
    print(df['vowel'].value_counts())
else:
    print("特征文件不存在，请先运行特征提取脚本")
    print("命令: python scripts/lesson_04_svm_vowel.py --mode train ...")

## 5. 数据可视化

可视化元音在 F1-F2 平面上的分布。

In [ ]:
if features_file.exists():
    plt.figure(figsize=(10, 8))
    
    for vowel in target_vowels:
        vowel_data = df[df['vowel'] == vowel]
        plt.scatter(vowel_data['F2'], vowel_data['F1'], 
                   label=vowel, alpha=0.6, s=50)
    
    plt.xlabel('F2 (Hz)', fontsize=12)
    plt.ylabel('F1 (Hz)', fontsize=12)
    plt.title('元音在 F1-F2 平面的分布', fontsize=14)
    plt.legend()
    plt.gca().invert_xaxis()  # F2 通常从右到左
    plt.gca().invert_yaxis()  # F1 通常从上到下
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. 训练 SVM 模型

In [ ]:
if features_file.exists():
    # 准备数据
    feature_columns = ['F1', 'F2', 'F3', 'duration']
    X = df[feature_columns].values
    y = df['vowel'].values
    
    # 划分数据集
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"训练集: {len(X_train)} 个样本")
    print(f"测试集: {len(X_test)} 个样本")
    
    # 初始化并训练 SVM
    svm_config = {
        'kernel': 'rbf',
        'C': 1.0,
        'gamma': 'auto',
        'random_state': 42
    }
    
    classifier = SVMClassifier(svm_config)
    classifier.build()
    classifier.fit(X_train, y_train)
    
    print("\n模型训练完成！")

## 7. 模型评估

In [ ]:
if features_file.exists():
    # 预测
    y_pred = classifier.predict(X_test)
    
    # 计算指标
    from sklearn.metrics import accuracy_score, classification_report
    
    accuracy = accuracy_score(y_test, y_pred)
    print(f"测试集准确率: {accuracy:.4f}")
    
    print("\n分类报告:")
    print(classification_report(y_test, y_pred))

## 8. 混淆矩阵可视化

In [ ]:
if features_file.exists():
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(y_test, y_pred, labels=target_vowels)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_vowels,
                yticklabels=target_vowels)
    plt.xlabel('预测标签', fontsize=12)
    plt.ylabel('真实标签', fontsize=12)
    plt.title('混淆矩阵', fontsize=14)
    plt.tight_layout()
    plt.show()

## 9. 决策边界可视化（2D）

使用 F1 和 F2 两个特征可视化决策边界。

In [ ]:
if features_file.exists():
    # 只使用 F1 和 F2 训练一个简化模型
    X_2d = df[['F1', 'F2']].values
    y_2d = df['vowel'].values
    
    X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
        X_2d, y_2d, test_size=0.2, random_state=42, stratify=y_2d
    )
    
    classifier_2d = SVMClassifier(svm_config)
    classifier_2d.build()
    classifier_2d.fit(X_train_2d, y_train_2d)
    
    # 创建网格
    h = 50  # 网格步长
    x_min, x_max = X_2d[:, 0].min() - 100, X_2d[:, 0].max() + 100
    y_min, y_max = X_2d[:, 1].min() - 100, X_2d[:, 1].max() + 100
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # 预测网格点
    Z = classifier_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    
    # 将预测结果转换为数值
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    le.fit(target_vowels)
    Z_numeric = le.transform(Z)
    Z_numeric = Z_numeric.reshape(xx.shape)
    
    # 绘制决策边界
    plt.figure(figsize=(12, 10))
    plt.contourf(xx, yy, Z_numeric, alpha=0.3, cmap='viridis')
    
    # 绘制数据点
    for i, vowel in enumerate(target_vowels):
        vowel_data = df[df['vowel'] == vowel]
        plt.scatter(vowel_data['F1'], vowel_data['F2'],
                   label=vowel, s=50, edgecolors='black', linewidth=0.5)
    
    plt.xlabel('F1 (Hz)', fontsize=12)
    plt.ylabel('F2 (Hz)', fontsize=12)
    plt.title('SVM 决策边界（F1-F2 平面）', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. 总结

本教程演示了：
1. ✅ 使用 Praat 提取语音特征
2. ✅ 构建元音分类数据集
3. ✅ 训练 SVM 分类器
4. ✅ 评估模型性能
5. ✅ 可视化分类结果和决策边界

### 下一步
- 尝试不同的核函数（linear, poly, sigmoid）
- 调整超参数（C, gamma）
- 添加更多特征（MFCC, 能量等）
- 使用交叉验证评估模型稳定性